In [ ]:
!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.1 MB/s eta 0:00:00


In [ ]:
import re
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from transformers import BartTokenizer, BartForConditionalGeneration

# Load pretrained model and tokenizer
model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)

# Extract video ID from YouTube URL
def extract_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")

# Fetch transcript with error handling
def get_transcript(video_url, lang='en'):
    video_id = extract_video_id(video_url)
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
        return transcript
    except (TranscriptsDisabled, NoTranscriptFound):
        print("Transcript not available or disabled for this video.")
        return None

# Clean the transcript
def clean_transcript(transcript):
    return " ".join([
        entry['text'] for entry in transcript
        if entry['text'].strip().lower() not in ['[music]', '[applause]', '[laughter]']
    ])

# Chunk the transcript for summarization
def chunk_text(text, max_tokens=450):
    words = text.split()
    return [" ".join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]

# Summarize a single chunk
def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=60,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Complete pipeline
def summarize_youtube_video(video_url):
    transcript_data = get_transcript(video_url)
    if not transcript_data:
        return "Transcript not available."

    print("Transcript fetched.")
    cleaned = clean_transcript(transcript_data)
    chunks = chunk_text(cleaned)

    print(f"Transcript split into {len(chunks)} chunk(s)...\n")
    all_summaries = []

    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i+1}...")
        summary = summarize(chunk)
        all_summaries.append(summary)

    final_summary = " ".join(all_summaries)
    print("\n===== FINAL SUMMARY =====\n")
    print(final_summary)
    return final_summary

# Example usage
youtube_url = "https://www.youtube.com/watch?v=KKNCiRWd_j0&t=102s"  # Replace this with a real YouTube URL
summarize_youtube_video(youtube_url)

Transcript fetched.
Transcript split into 8 chunk(s)...

Summarizing chunk 1...
Summarizing chunk 2...
Summarizing chunk 3...
Summarizing chunk 4...
Summarizing chunk 5...
Summarizing chunk 6...
Summarizing chunk 7...
Summarizing chunk 8...

===== FINAL SUMMARY =====

In 2010, just the mention of the phrase “AGI,” artificial general intelligence, would get you some strange looks and even a cold shoulder. People thought it was 50 years away or 100 years away, if it was even possible at all. Now, AI is beating humans at a whole range of tasks that people previously thought were way out of reach. "We are at an inflection point in the history of humanity," says Elon Musk. "We're headed towards the emergence of something that we are all struggling to describe, and yet we cannot control what we don't understand" "I think AI should best be understood as something like a new digital species," he says. In just 18 months, over a billion people have used large language models. AIs can now drive c

'In 2010, just the mention of the phrase “AGI,” artificial general intelligence, would get you some strange looks and even a cold shoulder. People thought it was 50 years away or 100 years away, if it was even possible at all. Now, AI is beating humans at a whole range of tasks that people previously thought were way out of reach. "We are at an inflection point in the history of humanity," says Elon Musk. "We\'re headed towards the emergence of something that we are all struggling to describe, and yet we cannot control what we don\'t understand" "I think AI should best be understood as something like a new digital species," he says. In just 18 months, over a billion people have used large language models. AIs can now drive cars, manage energy grids and even invent new molecules. Just a few years ago, people said that AI would never be creative. And yet today, millions of people enjoy meaningful conversations with AIs. AIs will be infinitely knowledgeable, and soon they\'ll be factually

In [ ]:
import re
import cv2
import subprocess
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    MarianMTModel, MarianTokenizer,
    pipeline, AutoTokenizer, AutoModelForQuestionAnswering
)
from pytube import YouTube

# Load summarization model
summarizer_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
summarizer_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

# Load QA model
qa_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
qa_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")
qa_pipeline = pipeline("question-answering", model=qa_model, tokenizer=qa_tokenizer)

def extract_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")

def get_transcript(video_url, lang='en'):
    video_id = extract_video_id(video_url)
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
        return transcript
    except (TranscriptsDisabled, NoTranscriptFound):
        print("Transcript not available or disabled for this video.")
        return None

def format_timestamp(seconds):
    minutes = int(seconds) // 60
    secs = int(seconds) % 60
    return f"{minutes:02}:{secs:02}"

def chunk_transcript(transcript, max_words=450):
    chunks = []
    current_chunk = []
    word_count = 0
    start_time = None

    for entry in transcript:
        text = entry['text'].strip().lower()
        if text in ['[music]', '[applause]', '[laughter]']:
            continue

        words = entry['text'].split()
        if not start_time:
            start_time = entry['start']

        if word_count + len(words) > max_words:
            end_time = current_chunk[-1]['start'] + current_chunk[-1]['duration']
            chunks.append((current_chunk, start_time, end_time))
            current_chunk = []
            word_count = 0
            start_time = entry['start']

        current_chunk.append(entry)
        word_count += len(words)

    if current_chunk:
        end_time = current_chunk[-1]['start'] + current_chunk[-1]['duration']
        chunks.append((current_chunk, start_time, end_time))

    return chunks

def summarize(text):
    inputs = summarizer_tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=60,
        num_beams=4,
        early_stopping=True
    )
    return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def translate_summary(summary_text, target_lang):
    model_name = f"Helsinki-NLP/opus-mt-en-{target_lang}"
    try:
        translator_tokenizer = MarianTokenizer.from_pretrained(model_name)
        translator_model = MarianMTModel.from_pretrained(model_name)
    except:
        return "\u26a0\ufe0f Translation model not available for this language."

    inputs = translator_tokenizer(summary_text, return_tensors="pt", truncation=True, padding=True)
    translated_ids = translator_model.generate(**inputs)
    translated_text = translator_tokenizer.decode(translated_ids[0], skip_special_tokens=True)
    return translated_text

def capture_frame(video_url, timestamp, output_path="frame.jpg"):
    yt = YouTube(video_url)
    stream = yt.streams.filter(file_extension='mp4').first()
    video_path = stream.download(filename="temp_video.mp4")

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
    success, frame = cap.read()

    if success:
        cv2.imwrite(output_path, frame)
        print(f"\u2705 Frame saved at {output_path}")
        return output_path
    else:
        print("\u274c Failed to extract frame.")
        return None

def summarize_youtube_video(video_url):
    transcript_data = get_transcript(video_url)
    if not transcript_data:
        return "Transcript not available.", [], ""

    video_id = extract_video_id(video_url)
    transcript_chunks = chunk_transcript(transcript_data)

    all_summaries = []
    full_summary_text = ""
    transcript_text = " ".join(entry["text"] for entry in transcript_data)

    print("\n📝 Generating summary...\n")
    summaries_only = []
    chunk_timestamps = []

    for i, (chunk, start, end) in enumerate(transcript_chunks):
        print(f"Summarizing chunk {i+1}...")
        text = " ".join(entry['text'] for entry in chunk)
        summary = summarize(text)
        summaries_only.append(summary)
        full_summary_text += f"{summary} "
        chunk_timestamps.append((start, end))

    print("\n===== FINAL SUMMARY (ENGLISH) =====\n")
    print(full_summary_text.strip())

    print("\n===== SUMMARY WITH TIMESTAMPS =====\n")
    for summary, (start, end) in zip(summaries_only, chunk_timestamps):
        timestamp_link = f"https://www.youtube.com/watch?v={video_id}&t={int(start)}s"
        label = f"[{format_timestamp(start)} - {format_timestamp(end)}]"
        clickable = f"{label} ({timestamp_link})"
        print(f"{clickable}\n{summary}\n")
        all_summaries.append(f"{clickable}\n{summary}")

    return full_summary_text.strip(), all_summaries, transcript_text

def run_chatbot(transcript_text):
    print("\n💬 Chatbot is ready! Ask questions about the video.")
    print("Type 'exit' to quit.\n")

    while True:
        question = input("You: ")
        if question.strip().lower() in ['exit', 'quit']:
            break

        response = qa_pipeline(question=question, context=transcript_text)
        print(f"🤖 Bot: {response['answer']}\n")

# Run pipeline
youtube_url = input("\U0001F3A5 Enter YouTube video URL: ")
english_summary, summary_with_timestamps, transcript_text = summarize_youtube_video(youtube_url)

# Ask for translation
translate_choice = input("\n\U0001F310 Do you want to translate the summary? (yes/no): ").strip().lower()
if translate_choice in ['yes', 'y']:
    print("\U0001F30D Example language codes: fr (French), hi (Hindi), ur (Urdu), de (German), es (Spanish)")
    target_lang = input("Enter target language code: ").strip().lower()
    translated = translate_summary(english_summary, target_lang)
    print(f"\n===== TRANSLATED SUMMARY ({target_lang.upper()}) =====\n")
    print(translated)
else:
    print("\n\u2705 Translation skipped.")

# Ask to capture frame
#frame_choice = input("\n\U0001F4F8 Do you want to capture a frame at a timestamp? (yes/no): ").strip().lower()
#if frame_choice in ['yes', 'y']:
#    try:
#        time_input = input("Enter timestamp in seconds (e.g., 120): ").strip()
#        timestamp = float(time_input)
 #       capture_frame(youtube_url, timestamp)
  #  except ValueError:
   #     print("\u26a0\ufe0f Invalid timestamp.")

# def capture_multiple_frames(video_url):
#     while True:
#         time_input = input("Enter timestamp in seconds (or type 'done' to finish): ").strip().lower()
#         if time_input == 'done':
#             break
#         try:
#             timestamp = float(time_input)
#             capture_frame(video_url, timestamp, output_path=f"frame_{int(timestamp)}.jpg")
#         except ValueError:
#             print("⚠️ Invalid input. Please enter a number or 'done'.")

from pytube import YouTube
from pytube.exceptions import VideoUnavailable, RegexMatchError
import cv2
import os

def capture_clip(video_url, timestamp, clip_duration=5, output_path="clip.mp4"):
    try:
        # Format timestamp for ffmpeg
        start_time = int(timestamp)
        duration = int(clip_duration)

        # Use yt-dlp to get the best video URL
        temp_video = "full_video.mp4"
        print("📥 Downloading video using yt-dlp...")
        subprocess.run([
            "yt-dlp",
            "-f", "best[ext=mp4]",
            "-o", temp_video,
            video_url
        ], check=True)

        # Cut clip with ffmpeg
        print("✂️ Cutting clip with ffmpeg...")
        subprocess.run([
            "ffmpeg",
            "-ss", str(start_time),
            "-i", temp_video,
            "-t", str(duration),
            "-c", "copy",
            output_path,
            "-y"  # Overwrite if exists
        ], check=True)

        print(f"✅ Clip saved as {output_path}")
        os.remove(temp_video)  # Clean up full video
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"❌ Error capturing clip: {e}")
        return None



clip_choice = input("\n🎬 Do you want to generate a video clip at a timestamp? (yes/no): ").strip().lower()
if clip_choice in ['yes', 'y']:
    try:
        time_input = input("Enter timestamp in seconds (e.g., 120): ").strip()
        timestamp = float(time_input)
        clip_path = capture_clip(youtube_url, timestamp)
        if clip_path:
            print(f"🎥 Saved clip: {clip_path}")
    except ValueError:
        print("⚠️ Invalid timestamp.")


# Ask for chatbot
chat_choice = input("\n\U0001F916 Do you want to ask questions about the video? (yes/no): ").strip().lower()
if chat_choice in ['yes', 'y']:
    run_chatbot(transcript_text)
# In the main pipeline





Device set to use cpu


🎥 Enter YouTube video URL: https://www.youtube.com/watch?v=Hu4Yvq-g7_Y

📝 Generating summary...

Summarizing chunk 1...
Summarizing chunk 2...
Summarizing chunk 3...
Summarizing chunk 4...
Summarizing chunk 5...
Summarizing chunk 6...

===== FINAL SUMMARY (ENGLISH) =====

Amanda Chu: "My life was a series of screens" Chu: I could spend hours on my phone every single day. She decided to get rid of the thing for a month. "I noticed that three curious things began to happen," she says. "It was like I could focus on things, not effortlessly, but with much more ease" to get to the bottom of what it takes to focus in a world of distraction. How does technology influenceour attention and our ability to focus? I want to start with the attention spans that we have. This is how we pay attention and how much control we have over our focus. It turns out that when we do work in front of a computer, we focus on one thing for just 40 seconds. CNN's John Sutter asked readers to make him bored for an h

In [ ]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.7 MB/s eta 0:00:00


In [ ]:
!pip install pytube

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
import re
import cv2
import subprocess
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    MarianMTModel, MarianTokenizer,
    pipeline, AutoTokenizer, AutoModelForQuestionAnswering
)
from pytube import YouTube

# Load summarization model
summarizer_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
summarizer_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

# Load QA model
qa_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
qa_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")
qa_pipeline = pipeline("question-answering", model=qa_model, tokenizer=qa_tokenizer)

def extract_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")

def get_transcript(video_url, lang='en'):
    video_id = extract_video_id(video_url)
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
        return transcript
    except (TranscriptsDisabled, NoTranscriptFound):
        print("Transcript not available or disabled for this video.")
        return None

def format_timestamp(seconds):
    minutes = int(seconds) // 60
    secs = int(seconds) % 60
    return f"{minutes:02}:{secs:02}"

def chunk_transcript(transcript, max_words=450):
    chunks = []
    current_chunk = []
    word_count = 0
    start_time = None

    for entry in transcript:
        text = entry['text'].strip().lower()
        if text in ['[music]', '[applause]', '[laughter]']:
            continue

        words = entry['text'].split()
        if not start_time:
            start_time = entry['start']

        if word_count + len(words) > max_words:
            end_time = current_chunk[-1]['start'] + current_chunk[-1]['duration']
            chunks.append((current_chunk, start_time, end_time))
            current_chunk = []
            word_count = 0
            start_time = entry['start']

        current_chunk.append(entry)
        word_count += len(words)

    if current_chunk:
        end_time = current_chunk[-1]['start'] + current_chunk[-1]['duration']
        chunks.append((current_chunk, start_time, end_time))

    return chunks

def summarize(text):
    inputs = summarizer_tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=60,
        num_beams=4,
        early_stopping=True
    )
    return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def translate_summary(summary_text, target_lang):
    model_name = f"Helsinki-NLP/opus-mt-en-{target_lang}"
    try:
        translator_tokenizer = MarianTokenizer.from_pretrained(model_name)
        translator_model = MarianMTModel.from_pretrained(model_name)
    except:
        return "\u26a0\ufe0f Translation model not available for this language."

    inputs = translator_tokenizer(summary_text, return_tensors="pt", truncation=True, padding=True)
    translated_ids = translator_model.generate(**inputs)
    translated_text = translator_tokenizer.decode(translated_ids[0], skip_special_tokens=True)
    return translated_text

def capture_frame(video_url, timestamp, output_path="frame.jpg"):
    yt = YouTube(video_url)
    stream = yt.streams.filter(file_extension='mp4').first()
    video_path = stream.download(filename="temp_video.mp4")

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
    success, frame = cap.read()

    if success:
        cv2.imwrite(output_path, frame)
        print(f"\u2705 Frame saved at {output_path}")
        return output_path
    else:
        print("\u274c Failed to extract frame.")
        return None

def summarize_youtube_video(video_url):
    transcript_data = get_transcript(video_url)
    if not transcript_data:
        return "Transcript not available.", [], ""

    video_id = extract_video_id(video_url)
    transcript_chunks = chunk_transcript(transcript_data)

    all_summaries = []
    full_summary_text = ""
    transcript_text = " ".join(entry["text"] for entry in transcript_data)

    print("\n📝 Generating summary...\n")
    summaries_only = []
    chunk_timestamps = []

    for i, (chunk, start, end) in enumerate(transcript_chunks):
        print(f"Summarizing chunk {i+1}...")
        text = " ".join(entry['text'] for entry in chunk)
        summary = summarize(text)
        summaries_only.append(summary)
        full_summary_text += f"{summary} "
        chunk_timestamps.append((start, end))

    print("\n===== FINAL SUMMARY (ENGLISH) =====\n")
    print(full_summary_text.strip())

    print("\n===== SUMMARY WITH TIMESTAMPS =====\n")
    for summary, (start, end) in zip(summaries_only, chunk_timestamps):
        timestamp_link = f"https://www.youtube.com/watch?v={video_id}&t={int(start)}s"
        label = f"[{format_timestamp(start)} - {format_timestamp(end)}]"
        clickable = f"{label} ({timestamp_link})"
        print(f"{clickable}\n{summary}\n")
        all_summaries.append(f"{clickable}\n{summary}")

    return full_summary_text.strip(), all_summaries, transcript_text

def run_chatbot(transcript_text):
    print("\n💬 Chatbot is ready! Ask questions about the video.")
    print("Type 'exit' to quit.\n")

    while True:
        question = input("You: ")
        if question.strip().lower() in ['exit', 'quit']:
            break

        response = qa_pipeline(question=question, context=transcript_text)
        print(f"🤖 Bot: {response['answer']}\n")

# Run pipeline
youtube_url = input("\U0001F3A5 Enter YouTube video URL: ")
english_summary, summary_with_timestamps, transcript_text = summarize_youtube_video(youtube_url)

# Ask for translation
translate_choice = input("\n\U0001F310 Do you want to translate the summary? (yes/no): ").strip().lower()
if translate_choice in ['yes', 'y']:
    print("\U0001F30D Example language codes: fr (French), hi (Hindi), ur (Urdu), de (German), es (Spanish)")
    target_lang = input("Enter target language code: ").strip().lower()
    translated = translate_summary(english_summary, target_lang)
    print(f"\n===== TRANSLATED SUMMARY ({target_lang.upper()}) =====\n")
    print(translated)
else:
    print("\n\u2705 Translation skipped.")

# Ask to capture frame
#frame_choice = input("\n\U0001F4F8 Do you want to capture a frame at a timestamp? (yes/no): ").strip().lower()
#if frame_choice in ['yes', 'y']:
#    try:
#        time_input = input("Enter timestamp in seconds (e.g., 120): ").strip()
#        timestamp = float(time_input)
 #       capture_frame(youtube_url, timestamp)
  #  except ValueError:
   #     print("\u26a0\ufe0f Invalid timestamp.")

# def capture_multiple_frames(video_url):
#     while True:
#         time_input = input("Enter timestamp in seconds (or type 'done' to finish): ").strip().lower()
#         if time_input == 'done':
#             break
#         try:
#             timestamp = float(time_input)
#             capture_frame(video_url, timestamp, output_path=f"frame_{int(timestamp)}.jpg")
#         except ValueError:
#             print("⚠️ Invalid input. Please enter a number or 'done'.")

from pytube import YouTube
from pytube.exceptions import VideoUnavailable, RegexMatchError
import cv2
import os

# Ask to capture multiple frames from the video
def capture_frame(video_url, timestamp, output_path="frame.jpg"):
    import subprocess
    import os
    import cv2

    temp_video = "temp_video.mp4"
    try:
        print("📥 Downloading video using yt-dlp...")
        subprocess.run([
            "yt-dlp",
            "-f", "best[ext=mp4]",
            "-o", temp_video,
            video_url
        ], check=True)

        print(f"📸 Capturing frame at {timestamp} seconds...")
        cap = cv2.VideoCapture(temp_video)
        cap.set(cv2.CAP_PROP_POS_MSEC, float(timestamp) * 1000)
        success, frame = cap.read()

        if success:
            cv2.imwrite(output_path, frame)
            print(f"✅ Frame saved as {output_path}")
        else:
            print("❌ Could not read frame at specified timestamp.")

        cap.release()
        os.remove(temp_video)

    except subprocess.CalledProcessError as e:
        print(f"❌ Error downloading or processing video: {e}")

def capture_multiple_frames(video_url):
    while True:
        time_input = input("Enter timestamp in seconds eg:120 etc (or type 'done' to finish): ").strip().lower()
        if time_input == 'done':
            break
        try:
            timestamp = float(time_input)
            output_name = f"frame_{int(timestamp)}.jpg"
            capture_frame(video_url, timestamp, output_path=output_name)
        except ValueError:
            print("⚠️ Invalid input. Please enter a number or 'done'.")

# Run screenshot capture
screenshot_choice = input("📸 Do you want to capture screenshots at specific timestamps? (yes/no): ").strip().lower()
if screenshot_choice in ['yes', 'y']:
    capture_multiple_frames(youtube_url)


# Ask for chatbot
chat_choice = input("\n\U0001F916 Do you want to ask questions about the video? (yes/no): ").strip().lower()
if chat_choice in ['yes', 'y']:
    run_chatbot(transcript_text)
# In the main pipeline





/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Device set to use cpu


🎥 Enter YouTube video URL: https://www.youtube.com/watch?v=KKNCiRWd_j0&t=102s

📝 Generating summary...

Summarizing chunk 1...
Summarizing chunk 2...
Summarizing chunk 3...
Summarizing chunk 4...
Summarizing chunk 5...
Summarizing chunk 6...
Summarizing chunk 7...
Summarizing chunk 8...

===== FINAL SUMMARY (ENGLISH) =====

In 2010, just the mention of the phrase “AGI,” artificial general intelligence, would get you some strange looks and even a cold shoulder. People thought it was 50 years away or 100 years away, if it was even possible at all. Now AI is beating humans at a whole range of tasks that people previously thought were way out of reach.  AI should best be understood as something like a new digital species, says Tim Berners-Lee. He predicts that we'll come to see them as digital companions, new partners in the journeys of all our lives. "We're headed towards the emergence of something that we are all struggling to describe, and yet we cannot control," he says. In just 18 mon

In [ ]:
pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.0 MB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.enums import TA_CENTER
import os

def generate_pdf_report(video_url, english_summary, translated_summary=None, screenshot_paths=None):
    if screenshot_paths is None:
        screenshot_paths = []

    # Prepare file name and doc
    pdf_filename = "YouTube_Video_Report.pdf"
    doc = SimpleDocTemplate(pdf_filename, pagesize=A4, rightMargin=30, leftMargin=30, topMargin=30, bottomMargin=18)

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='CenterHeading', alignment=TA_CENTER, fontSize=16, spaceAfter=12))
    flowables = []

    # 1. Title
    flowables.append(Paragraph("🎬 YouTube Video Summary Report", styles['CenterHeading']))
    flowables.append(Spacer(1, 12))

    # 2. Clickable Link
    video_link = f'<link href="{video_url}">{video_url}</link>'
    flowables.append(Paragraph("📎 <b>Video Link:</b>", styles["Normal"]))
    flowables.append(Paragraph(video_link, styles["Normal"]))
    flowables.append(Spacer(1, 12))

    # 3. English Summary
    flowables.append(Paragraph("📝 <b>English Summary:</b>", styles["Normal"]))
    flowables.append(Paragraph(english_summary.replace("\n", "<br/>"), styles["Normal"]))
    flowables.append(Spacer(1, 12))

    # 4. Translated Summary (optional)
    if translated_summary:
        flowables.append(Paragraph("🌐 <b>Translated Summary:</b>", styles["Normal"]))
        flowables.append(Paragraph(translated_summary.replace("\n", "<br/>"), styles["Normal"]))
        flowables.append(Spacer(1, 12))

    # 5. Screenshots
    if screenshot_paths:
        flowables.append(Paragraph("📸 <b>Captured Screenshots:</b>", styles["Normal"]))
        flowables.append(Spacer(1, 6))
        for path in screenshot_paths:
            if os.path.exists(path):
                img = Image(path, width=5.5*inch, height=3.0*inch)
                flowables.append(img)
                flowables.append(Spacer(1, 12))
            else:
                flowables.append(Paragraph(f"❌ Image not found: {path}", styles["Normal"]))
    else:
        flowables.append(Paragraph("📸 <i>No screenshots were captured.</i>", styles["Normal"]))

    # Build PDF
    doc.build(flowables)
    print(f"\n📄 PDF report generated: {pdf_filename}")


In [ ]:
# Collect all screenshots (if any)
screenshot_paths = [f for f in os.listdir() if f.startswith("frame_") and f.endswith(".jpg")]

# Ask if translated summary was created
translated_summary = translated if 'translated' in locals() else None

# Generate the final PDF report
generate_pdf_report(youtube_url, english_summary, translated_summary, screenshot_paths)



📄 PDF report generated: YouTube_Video_Report.pdf
